# RailLens — Feature Engineering + ML

**Goal:** predict train ticket price while demonstrating the Feature Engineering syllabus.

In [ ]:
import os, sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
sys.path.append("../src")
from feature_engineering import add_features
df=pd.read_csv("../data/train-ticket-price.csv")
df.shape, df.head()

## 1. Data audit
Check schema, missingness, duplicates, and target distribution before engineering.

In [ ]:
print(df.info())
display(df.describe(include="all").T)
print("Duplicates:", df.duplicated().sum())
display(df.isna().mean().sort_values(ascending=False).head(15))
df["price"].hist(bins=50, figsize=(9,4)); plt.title("Ticket price distribution"); plt.show()

## 2. Feature engineering
Parse timestamps, create temporal features, duration, advance-booking time, journey/route interactions and frequency features.

In [ ]:
df2=add_features(df)
print(df2.shape)
display(df2.head())
display(df2.dtypes)

## 3. Train/test split + leakage-safe pipeline
The split occurs before learned preprocessing. Imputation, scaling and one-hot encoding are fitted only on the training partition.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.ensemble import RandomForestRegressor

df_model=df.drop_duplicates().dropna(subset=["price"])
tr,te=train_test_split(df_model,test_size=0.2,random_state=42)
Xtr,Xte=add_features(tr),add_features(te)
ytr,yte=tr.price,te.price
cat=[c for c in Xtr.columns if Xtr[c].dtype=="object" or str(Xtr[c].dtype).startswith("category")]
num=[c for c in Xtr.columns if c not in cat]
pre=ColumnTransformer([
("num",Pipeline([("imp",SimpleImputer(strategy="median",add_indicator=True)),("scale",RobustScaler())]),num),
("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("ohe",OneHotEncoder(handle_unknown="ignore",min_frequency=3))]),cat)
])
pipe=Pipeline([("pre",pre),("model",RandomForestRegressor(n_estimators=200,max_depth=22,min_samples_leaf=2,n_jobs=-1,random_state=42))])
pipe.fit(Xtr,ytr)
pred=pipe.predict(Xte)
print("MAE",mean_absolute_error(yte,pred))
print("RMSE",mean_squared_error(yte,pred)**0.5)
print("R2",r2_score(yte,pred))

## 4. Cross-validation
Use K-fold CV to estimate stability rather than trusting one split.

In [ ]:
cv=KFold(n_splits=5,shuffle=True,random_state=42)
scores=-cross_val_score(pipe,Xtr,ytr,cv=cv,scoring="neg_mean_absolute_error",n_jobs=-1)
print("CV MAE:",scores)
print("Mean CV MAE:",scores.mean())

## 5. Filter + embedded feature selection
Use variance filtering and mutual information on a numeric feature subset; tree-based importance is shown after the full pipeline.

In [ ]:
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression
num_raw=tr.select_dtypes(include=np.number).copy()
num_raw=num_raw.drop(columns=["price"],errors="ignore").replace([np.inf,-np.inf],np.nan).fillna(num_raw.median(numeric_only=True))
vt=VarianceThreshold(threshold=0.0)
vt.fit(num_raw)
print("Numeric columns:",len(num_raw.columns),"after variance filter:",vt.get_support().sum())
mi=mutual_info_regression(num_raw,ytr,random_state=42)
mi_rank=pd.Series(mi,index=num_raw.columns).sort_values(ascending=False)
display(mi_rank.head(15))

## 6. Permutation importance
Model-agnostic importance on the held-out set.

In [ ]:
from sklearn.inspection import permutation_importance
r=permutation_importance(pipe,Xte,yte,n_repeats=5,random_state=42,n_jobs=-1,scoring="neg_mean_absolute_error")
imp=pd.Series(r.importances_mean,index=Xte.columns).sort_values(ascending=False)
display(imp.head(20))

## 7. PCA demonstration
PCA is demonstrated separately because it is not automatically applied to the production pipeline: tree models and one-hot encoded features have different interpretability trade-offs.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
Z=StandardScaler().fit_transform(num_raw)
pca=PCA().fit(Z)
cum=np.cumsum(pca.explained_variance_ratio_)
plt.figure(figsize=(8,4)); plt.plot(range(1,len(cum)+1),cum,marker="o")
plt.xlabel("Number of components"); plt.ylabel("Cumulative explained variance"); plt.grid(); plt.show()
print("Components for 90% variance:",np.argmax(cum>=0.90)+1)

## 8. Residual analysis

In [ ]:
res=yte.to_numpy()-pred
plt.figure(figsize=(8,4)); plt.scatter(pred,res,s=8,alpha=.25); plt.axhline(0,linestyle="--")
plt.xlabel("Predicted price"); plt.ylabel("Residual"); plt.title("Residual plot"); plt.show()